# Notebook 0 — Guided tour: the project, the concepts, the first code

This notebook assumes **no prior knowledge** of retrieval or of LLM
evaluation. It moves step by step:

1. what the project builds, and the problem it tackles;
2. the concepts and the vocabulary, on a five-line toy example;
3. the first real dataset, loaded and explored;
4. only then, the code in `src/` that did all of that, read block by block;
5. an overview of the datasets of the later phases;
6. experiments to run yourself, interview questions, a glossary.

Two reading rules, valid for every notebook in the project:

- **Nothing is reimplemented here.** Everything that runs is imported from
  `src/llm_testbench/` — the project's real code, the code that is tested and deployed.
  When we want to read it, we display it with `show_source()` (a thin wrapper around
  `inspect.getsource()` that adds syntax highlighting). What you read is what
  runs.
- **Every technical term is defined at its first appearance** (and summarized in the
  glossary at the very bottom). If a term is not defined, that is a bug in the notebook.

Two run modes:
- **full** (default): downloads the real dataset (~4.5 MB, only once);
- **sample** (`LLM_TESTBENCH_SAMPLE=1`, what the CI does): a few lines of
  local data, zero network — checks that everything runs, produces no usable
  number.

## Part 1 — What we are building, and the problem we are tackling

### The system

An LLM (*Large Language Model* — GPT, Claude, Mistral…) only knows what it saw during
training. Ask it a question about **your** documents — a contract, an internal
doc, a report — and all it can do is make things up. The standard remedy is called **RAG**
(*Retrieval-Augmented Generation*, i.e. generation augmented by search) and comes
in three steps:

```
question → [1. RETRIEVE the relevant passages from the documents]
         → [2. Give them to the LLM as context]
         → [3. GENERATE an answer grounded in those passages, with citations]
```

This project builds that complete system — then extends it: querying SQL databases,
APIs, images (phase 4), Kubernetes deployment, observability, costs (phases 1 and 5).
On the infra side, you are on familiar ground. What is new is everything else — and it
starts here.

### The problem: how do you know whether it works?

Imagine the RAG above plugged into 5,000 medical articles. You ask it "do
statins cause muscle pain?" and it answers with three plausible paragraphs,
with citations. **Is it any good?** You are not a doctor. You
read the citation: it looks related. You test ten questions, it "looks
fine", you move on.

Three months later, you change a parameter (the size of the text chunks, the
model, whatever). Is the system better or worse? **You have no
way of knowing.** That is the state of most RAG projects in companies, and
it is exactly what this project refuses to be: a system judged "by eye".

### The solution: public evaluation datasets, already annotated

To measure, you need a **ground truth**: for each
test question, the right answer — established in advance, by humans. Building that
yourself is very expensive (weeks of annotation) and requires domain
expertise.

Hence the project's structuring choice: rely on **public datasets**
built by research teams, where this annotation work is
already done. These datasets serve as **testbenches** (*benchmarks*): the scientific
community measures its systems on them and **publishes its scores** in papers — what we
will call "the literature".

The benefit is twofold, and the second point is the more important one:

1. **Measure objectively** every technique we will add (a technique without a measured
   gain will be removed);
2. **Calibrate ourselves**: if our implementation of a classic algorithm comes out far
   below the published score for that same algorithm on that same dataset, *our code is
   wrong* — and we know it the same day. Without that reference point, you can optimize
   a broken system for three weeks without noticing.

Two limitations, accepted and written in the README:

- **No real users.** Everything that assumes production traffic will be simulated
  (phase 5) — and we will document what the simulation does not capture.
- **Contamination**: these public datasets are old and famous, so probably
  present in the training data of the LLMs we will evaluate. A model that has already
  "seen" the answers gets inflated scores. We cannot prevent it; we can
  measure it (phase 2), and being able to talk about it is a real marker of seniority in
  an interview.

## Part 2 — The concepts, on a toy example

### Retrieval

**Retrieval** ("search", in the search-engine sense) is step 1 of RAG:
finding, among thousands of documents, the few that make it possible to answer
the question. It is half of a RAG's quality: if the right passages are not
retrieved, the LLM cannot answer well, however good it is.

So it is the **first thing we will want to measure** (phase 2), and the project's first
dataset is a retrieval dataset. Let's see what such a dataset looks like, in a
tiny version.

### A retrieval evaluation dataset, in five lines

Three one-sentence "documents", two questions:

| doc id | text |
|---|---|
| D1 | Statins can cause muscle pain in some patients. |
| D2 | The Eiffel Tower is 330 meters tall. |
| D3 | A randomized trial found no increase in myalgia on statins. |

| question id | text |
|---|---|
| Q1 | Do statins cause muscle pain? |
| Q2 | How tall is the Eiffel Tower? |

A human now judges, for each question, which documents are relevant:

| question | document | relevant? |
|---|---|---|
| Q1 | D1 | yes (1) |
| Q1 | D3 | yes (1) |
| Q2 | D2 | yes (1) |

These judgments are the dataset's ground truth. A perfect retrieval system,
queried with Q1, must return D1 and D3 — and not D2. We can now **count**
its successes and failures: that is what measuring retrieval means.

### The official vocabulary: corpus, queries, qrels

These three tables carry names established by 30 years of research in *information
retrieval* (IR). We keep this vocabulary as is — it is the one used in the papers we
will compare our scores against, and in interviews:

- the **corpus**: the set of documents we search in (our D1-D3);
- the **queries**: the test requests (our Q1-Q2). Depending on the dataset, they are
  questions, keywords or claims;
- the **qrels** (*query relevance judgments*): the
  third table — for each query, the documents judged relevant, with a relevance
  level. Often binary (1 = relevant); sometimes **graded** (0 = not
  relevant, 1 = somewhat, 2 = very — useful when not all relevant documents are
  equal).

A detail that matters: the qrels list only the **judged** documents. Nobody
checked D2 for Q1 — the absence of a judgment is treated as "not relevant"; this is
a standard approximation in the field (judging every pair is impossible as soon as
the corpus grows).

### Splits: train and test

Datasets are divided into **splits** (subsets): typically **train** (to
tune your system) and **test** (to measure it). The hygiene rule, identical to classical
ML: you never tune on the test split, otherwise the score no longer predicts anything
— you would have "memorized the test". Some datasets even keep the test
answers secret (submission to an online leaderboard); you then evaluate on the
validation split.

### SciFact, our first real dataset

The project starts with **SciFact**, a scientific retrieval dataset created by
the AI2 institute and distributed in **BEIR** — a collection of some fifteen retrieval
datasets in a uniform format, which has become the de facto standard for comparing
search systems (it is "the" retrieval benchmark cited everywhere).

In SciFact:

- the **corpus**: 5,183 *abstracts* of biomedical papers;
- the **queries**: scientific **claims**, not questions —
  for example *"1 in 5 million in UK have abnormal PrP positivity"*;
- the **qrels**: for each claim, the abstract(s) that **support or
  refute** it (~1.1 relevant documents per query: almost always just one).

Why this one first? It is the simplest to validate in the whole collection:
small (4.5 MB — iterations are instant), near-binary qrels, and plenty of published
scores to calibrate against. Claims rather than questions is
also a good robustness test: many systems are tuned for questions.

## Part 3 — The real dataset, loaded and explored

We look at the data first; we will open the hood on the code in part 4.

The next two cells are the **configuration**: constants you
can change before re-running the whole notebook ("Run All") — this is the
experimentation mode of every notebook in the project, no code writing required.

In [1]:
# --- Configuration: edit, then "Run All" ---
import os

# Sample mode: tiny local data, zero network (this is the CI mode).
# The environment variable LLM_TESTBENCH_SAMPLE=1 enables it.
SAMPLE = os.environ.get("LLM_TESTBENCH_SAMPLE", "0") == "1"

# In sample mode: maximum number of queries kept.
MAX_QUERIES = 5

# Split loaded below: "test" or "train".
SPLIT = "test"

# Index of the query detailed in the outputs below (0 = the first one).
SHOW_QUERY_INDEX = 0

print(f"sample mode: {SAMPLE}")

sample mode: True


In [2]:
import inspect
import statistics

from IPython.display import Markdown, display

from llm_testbench import __version__
from llm_testbench.config import Settings, find_repo_root
from llm_testbench.eval import types as eval_types
from llm_testbench.eval.loaders import BeirLoader
from llm_testbench.eval.loaders import beir as beir_module

# The notebook runs from notebooks/phase-0/: paths are anchored at the repository root.
ROOT = find_repo_root()
settings = Settings(
    # Where downloaded datasets are cached (never versioned in git).
    data_dir=ROOT / "data" / "cache",
    # In sample mode, we read the same tiny fixtures as the tests.
    fixtures_dir=(ROOT / "tests" / "fixtures") if SAMPLE else None,
    sample=SAMPLE,
)


def show_source(*objects: object) -> None:
    """Display the real source of objects from src/ with syntax highlighting.

    inspect.getsource() returns plain text; we re-emit it in a Markdown
    ```python block that the front end (VS Code, Jupyter, GitHub) highlights itself.
    Display only — no project logic lives here.
    """
    blocks = [f"```python\n{inspect.getsource(obj)}```" for obj in objects]
    display(Markdown("\n".join(blocks)))


print(f"llm_testbench {__version__} — root: {ROOT}")
print(f"data source: {'local fixtures' if SAMPLE else 'Hugging Face (cache data/cache/)'}")

llm_testbench 0.1.0 — root: /Users/christophe/Roadmaps/roadmap-ai-engineer
data source: local fixtures


### Loading SciFact

`BeirLoader` is the project code (`src/llm_testbench/eval/loaders/beir.py`) that
downloads SciFact and turns it into a validated Python object. We run it, then
`describe()` summarizes what was loaded:

In [3]:
loader = BeirLoader("scifact", settings=settings)
dataset = loader.load(SPLIT, max_queries=MAX_QUERIES if SAMPLE else None)
dataset.describe()

{'dataset': 'beir/scifact',
 'split': 'test',
 'documents': 6,
 'queries': 3,
 'judgments': 4,
 'sampled': False}

Reading the summary, field by field:

- `documents`: the corpus size. In full mode: 5,183 abstracts.
- `queries`: the claims **of the requested split** — 300 in test. (The dataset contains
  1,109 in total; we will see in part 4 why this filtering is an important pitfall.)
- `judgments`: the total number of relevance judgments (qrels rows).
- `sampled`: `True` if the dataset was truncated (sample mode). This flag travels
  **with the data**: any downstream code can refuse to produce an
  official number from a truncated dataset. We will come back to this.

Each dataset also carries its **provenance** — where it comes from, under which license,
where to read more. Every future number of the testbench will have to be traceable back
to its source:

In [4]:
# Provenance travels with the data: enough to trace every future number.
dataset.info.model_dump()

{'key': 'beir/scifact',
 'name': 'SciFact (BEIR)',
 'homepage': 'https://github.com/allenai/scifact',
 'license': 'cc-by-sa-4.0 (claim of the BEIR packaging, see ADR 001)',
 'source': 'hf:BeIR/scifact + hf:BeIR/scifact-qrels'}

### A real query, its qrels, its documents

The corpus / queries / qrels triplet from part 2, for real (change
`SHOW_QUERY_INDEX` at the top to see others):

In [5]:
query = dataset.queries[SHOW_QUERY_INDEX]
print(f"query {query.query_id!r}: {query.text}\n")
print(f"raw qrels: {dataset.qrels[query.query_id]}\n")

for doc_id in sorted(dataset.relevant_doc_ids(query.query_id)):
    doc = dataset.document_by_id(doc_id)
    excerpt = doc.text[:280] + ("…" if len(doc.text) > 280 else "")
    print(f"--- doc {doc.doc_id} — {doc.title}")
    print(f"    {excerpt}\n")

query '1': 0-dimensional biomaterials show inductive properties.

raw qrels: {'4983': 1, '5836': 1}

--- doc 4983 — Microstructural development of human brain
    Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities.

--- doc 5836 — Induction of myelodysplasia by myeloid-derived suppressor cells
    Myelodysplastic syndromes are age-dependent stem cell malignancies that share biological features of activated adaptive immune response and ineffective hematopoiesis.



Note the shape: the query is a **claim**; the document judged relevant is
the abstract that makes it possible to support or refute it. In phase 2, we will ask our
system to find this document among the 5,183 — without showing it the qrels, of
course — and we will count its successes.

A quick look at the dataset's profile (document sizes, judgment density) —
useful for sizing what comes next (abstracts are short; long documents
and chunking are the subject of QASPER, phase 3):

In [6]:
doc_words = [len(d.text.split()) for d in dataset.documents]
judgments_per_query = [len(j) for j in dataset.qrels.values()]

print(f"documents: {len(doc_words)}")
print(
    "words/document: "
    f"min {min(doc_words)}, median {statistics.median(doc_words):.0f}, max {max(doc_words)}"
)
print(
    "judgments/query: "
    f"mean {statistics.mean(judgments_per_query):.2f}, max {max(judgments_per_query)}"
)

documents: 6
words/document: min 16, median 20, max 23
judgments/query: mean 1.33, max 2


### The testbench's first reflex: compare yourself to the literature

The project's founding reflex: whenever a number can be checked against a
published source, we do it — and **a gap is a bug, not a result**.

We do not have a score yet (metrics arrive in phase 2), but we already have
numbers to check: the dataset volumes. The BEIR paper publishes for SciFact:
5,183 documents, 300 test queries, ~1.1 judgments per query. The next cell
checks — and **fails loudly** on any gap, including in the CI.

It also mentions the reference point we will aim for in phase 2. Two terms to know:

- **BM25**: the classic keyword search algorithm (1990s, at the heart
  of Elasticsearch). No neural network, and yet still formidable — it is
  THE baseline to beat in retrieval, and many "modern" systems do not beat
  it.
- **nDCG@10**: the standard retrieval metric. Intuition: between 0 and 1, it
  rewards a system that places the relevant documents **at the top** of the first 10
  results. Precise definition in phase 2, with our implementation.

The literature says: BM25 scores ~0.67 nDCG@10 on SciFact. When we implement
BM25 in phase 2, we must reproduce that number. Far below → our code is wrong.
Far above → our code is wrong *too* (test leakage, miscomputed metric…).

In [7]:
if dataset.sampled or SAMPLE:
    print("Sample mode: numbers NOT publishable, literature check not applicable.")
    print(dataset.describe())
else:
    expected = {"documents": 5183, "queries": 300}
    actual = dataset.describe()
    for key, value in expected.items():
        assert actual[key] == value, f"{key}: {actual[key]} != {value} (BEIR paper)"
    print("OK — volumes match the BEIR paper:", {k: actual[k] for k in expected})
    print("Phase 2 reference point: BM25 ≈ 0.67 nDCG@10 on SciFact (BEIR paper, table 2).")

Sample mode: numbers NOT publishable, literature check not applicable.
{'dataset': 'beir/scifact', 'split': 'test', 'documents': 6, 'queries': 3, 'judgments': 4, 'sampled': False}


### What SciFact measures — and what it does not

**Measures**: the ability to find, among 5,183 abstracts, the one that supports or
refutes a scientific claim. Ideal validation of a retrieval mechanism.

**Does not measure**: generation (no written answer expected — we only test the
"retrieve" step), multi-hop (a single document is almost always enough —
*multi-hop*: questions that require **combining several documents**), refusal
(every query has its truth — nothing tests "I don't know"), scaling
(small corpus). Each of these blind spots has its dedicated dataset in the testbench —
that is precisely why several are needed. Overview in part 5.

## Part 4 — Under the hood: the code that did this

Everything above was produced by two files in `src/` that we will read in
order:

1. `eval/types.py` — the objects that **represent** a dataset (the contract);
2. `eval/loaders/beir.py` — the code that **produces** these objects from the
   distributed files (the loader).

The classes are written with **Pydantic**, the standard data validation library
in Python: you declare the fields and their types, and Pydantic **validates at
construction** — an invalid object cannot exist. That is the philosophy of everything
that follows. The three building blocks first:

In [8]:
show_source(eval_types.Document, eval_types.Query, eval_types.DatasetInfo)

```python
class Document(BaseModel):
    model_config = ConfigDict(frozen=True)

    doc_id: str
    text: str
    title: str = ""
    metadata: dict[str, str] = Field(default_factory=dict)
```
```python
class Query(BaseModel):
    model_config = ConfigDict(frozen=True)

    query_id: str
    text: str
    metadata: dict[str, str] = Field(default_factory=dict)
```
```python
class DatasetInfo(BaseModel):
    """Identity card of a dataset: enough to trace the provenance of every number."""

    model_config = ConfigDict(frozen=True)

    key: str
    """Short identifier used everywhere in the testbench (e.g. ``beir/scifact``)."""

    name: str
    homepage: str
    license: str
    """License as displayed by the distribution source — see the verification of
    2026-08-26 in docs/: for BEIR, it is the claim of the packaging, not an upstream
    guarantee."""

    source: str
    """Where the data is actually loaded from (Hugging Face repositories, URL...)."""
```

Three discreet but important choices:

- **`frozen=True`**: the objects are immutable (any modification after creation is
  an error). An evaluation dataset is a reference — no downstream code should
  be able to quietly "fix" a document.
- **Identifiers are `str` everywhere**, even when the source delivers them as
  integers. The full story below — it is a real pitfall.
- **`DatasetInfo`**: the identity card seen in part 3 (exact source, license
  *as displayed* — an important nuance: for BEIR it is the claim of the
  packaging, not a guarantee from the original author; checked in
  `docs/datasets-verification-2026-08-26.md`).

Now the main container. Focus on `_check_referential_integrity`,
which is the real reason this class exists:

In [9]:
show_source(eval_types.RetrievalDataset)

```python
class RetrievalDataset(BaseModel):
    model_config = ConfigDict(frozen=True)

    info: DatasetInfo
    split: str
    documents: tuple[Document, ...]
    queries: tuple[Query, ...]
    qrels: Qrels

    sampled: bool = False
    """True if the dataset was truncated (sample mode). A sampled dataset checks the
    mechanics; it NEVER produces a publishable number."""

    @model_validator(mode="after")
    def _check_referential_integrity(self) -> "RetrievalDataset":
        doc_ids = {d.doc_id for d in self.documents}
        query_ids = {q.query_id for q in self.queries}

        if len(doc_ids) != len(self.documents):
            raise ValueError("duplicate doc_id in the corpus")
        if len(query_ids) != len(self.queries):
            raise ValueError("duplicate query_id in the queries")

        orphan_queries = set(self.qrels) - query_ids
        if orphan_queries:
            raise ValueError(f"qrels reference missing queries: {sorted(orphan_queries)[:5]}")
        orphan_docs = {doc_id for judged in self.qrels.values() for doc_id in judged} - doc_ids
        if orphan_docs:
            raise ValueError(f"qrels reference missing documents: {sorted(orphan_docs)[:5]}")

        unjudged = query_ids - set(self.qrels)
        if unjudged:
            raise ValueError(
                f"queries without any judgment: {sorted(unjudged)[:5]} — "
                "the loader must filter out queries outside the split"
            )
        return self

    def document_by_id(self, doc_id: str) -> Document:
        try:
            return self._documents_by_id[doc_id]
        except KeyError:
            raise KeyError(f"unknown document: {doc_id!r}") from None

    def relevant_doc_ids(self, query_id: str, min_relevance: int = 1) -> set[str]:
        return {
            doc_id for doc_id, rel in self.qrels.get(query_id, {}).items() if rel >= min_relevance
        }

    @property
    def _documents_by_id(self) -> dict[str, Document]:
        # Lazy index, cached on the instance (allowed despite frozen since outside the model).
        cached: dict[str, Document] | None = self.__dict__.get("_doc_index")
        if cached is None:
            cached = {d.doc_id: d for d in self.documents}
            object.__setattr__(self, "_doc_index", cached)
        return cached

    def describe(self) -> dict[str, int | str | bool]:
        """Compact summary — what the notebook displays, what the harness will log."""
        judgments = sum(len(j) for j in self.qrels.values())
        return {
            "dataset": self.info.key,
            "split": self.split,
            "documents": len(self.documents),
            "queries": len(self.queries),
            "judgments": judgments,
            "sampled": self.sampled,
        }
```

### The validator, check by check

`@model_validator(mode="after")`: Pydantic runs this method at every
construction. It rejects:

1. **Duplicate ids.** Two documents `"4983"` → which one is right? Error.
2. **Orphan qrels** — a judgment that points to a query or a document that is
   absent. That is the signature of a **failed join**: the three tables (corpus,
   queries, qrels) come from separate files, and if you assemble them badly, you get
   exactly that.
3. **A query without any judgment.** The subtlest one. Remember: the metrics
   count the relevant documents retrieved. A query without qrels has *nothing* to
   retrieve — its score will be 0 whatever the system does. Letting it through means
   diluting the dataset average **silently**. SciFact distributes 1,109 queries of which
   300 are judged in test: without filtering, every score would be divided by ~3.7,
   without a single error message. The validator makes this error *impossible*.

The general principle: **fail loudly, immediately, at the place of the fault** —
not three phases later in an odd score that will take days to explain.

And the `sampled` flag: set by the loader when the dataset is truncated, it makes a
sample dataset **self-declared** as unfit for official numbers. It is not a
team convention (which gets forgotten), it is carried by the data (which does not
forget).

### Where the files come from: Hugging Face and the BEIR pitfalls

The BEIR datasets are distributed on the **Hugging Face Hub** — the "GitHub of models
and datasets", from which the Python library `datasets` knows how to download. Three
peculiarities, discovered while verifying (they are written down nowhere):

1. Each dataset lives in **two repositories**: `BeIR/scifact` (the corpus and the
   queries) and `BeIR/scifact-qrels` (the judgments). Loading a dataset = joining three
   tables coming from two repositories.
2. The ids do **not have the same type** depending on the table: `str` in the corpus,
   integers in the qrels. In Python, `4983 == "4983"` is `False` — **silently**: without
   systematic conversion, the join matches nothing, all qrels become
   orphans — and without our validator, the recall would simply be wrong.
3. The queries file contains the 1,109 queries of **all** splits mixed together —
   hence the filtering seen above.

Here is the loader. Its only intelligence of its own: choosing the **source** of the raw
rows (Hugging Face, or local files) — everything else is delegated:

In [10]:
show_source(BeirLoader)

```python
class BeirLoader(RetrievalLoader):
    def __init__(self, dataset: str = "scifact", settings: Settings | None = None) -> None:
        if dataset not in BEIR_DATASETS:
            raise ValueError(
                f"BEIR dataset not wired: {dataset!r} (known: {sorted(BEIR_DATASETS)})"
            )
        self._dataset = dataset
        self._settings = settings if settings is not None else get_settings()

    @property
    def info(self) -> DatasetInfo:
        return BEIR_DATASETS[self._dataset]

    def load(self, split: str = "test", max_queries: int | None = None) -> RetrievalDataset:
        corpus_rows, query_rows, qrels_rows = self._load_rows(split)
        return build_retrieval_dataset(
            info=self.info,
            split=split,
            corpus_rows=corpus_rows,
            query_rows=query_rows,
            qrels_rows=qrels_rows,
            max_queries=max_queries,
        )

    # ------------------------------------------------------------------ sources

    def _load_rows(self, split: str) -> tuple[list[Row], list[Row], list[Row]]:
        fixtures_dir = self._settings.fixtures_dir
        if fixtures_dir is not None:
            return self._load_rows_from_fixtures(fixtures_dir, split)
        return self._load_rows_from_hf(split)

    def _load_rows_from_hf(self, split: str) -> tuple[list[Row], list[Row], list[Row]]:
        from datasets import load_dataset

        cache_dir = str(self._settings.data_dir / "hf")
        repo = f"BeIR/{self._dataset}"
        corpus = load_dataset(repo, "corpus", split="corpus", cache_dir=cache_dir)
        queries = load_dataset(repo, "queries", split="queries", cache_dir=cache_dir)
        qrels = load_dataset(f"{repo}-qrels", split=split, cache_dir=cache_dir)
        return list(corpus), list(queries), list(qrels)

    def _load_rows_from_fixtures(
        self, fixtures_dir: Path, split: str
    ) -> tuple[list[Row], list[Row], list[Row]]:
        base = fixtures_dir / "beir" / self._dataset
        return (
            _read_json_rows(base / "corpus.json"),
            _read_json_rows(base / "queries.json"),
            _read_json_rows(base / f"qrels_{split}.json"),
        )
```

Why two sources? Because of a testing principle of the project: **the CI runs
offline** (no network, no API key, no flakiness). The tests and sample
mode read **fixtures** — tiny hand-made data files
(6 documents, 5 queries), versioned in `tests/fixtures/`, that reproduce the
exact structure of the real files, pitfalls included (the ids are integers there too
on the qrels side).

The important part: the join itself is a **pure function** (same inputs → same
output, no network access), the same for the fixtures and the real data. So we do not
test a simplified copy of the logic — we replay the real logic on
tiny data. Here it is, in three blocks:

In [11]:
show_source(beir_module.build_retrieval_dataset)

```python
def build_retrieval_dataset(
    info: DatasetInfo,
    split: str,
    corpus_rows: list[Row],
    query_rows: list[Row],
    qrels_rows: list[Row],
    max_queries: int | None = None,
) -> RetrievalDataset:
    """Join corpus + queries + qrels into a validated dataset.

    Pure and independent of the source of the rows: it is what the offline tests
    cover, the same function that runs on the real data.
    """
    qrels: Qrels = {}
    for row in qrels_rows:
        # The qrels ids sometimes arrive typed as int on the HF side: everything becomes str.
        query_id, doc_id = str(row["query-id"]), str(row["corpus-id"])
        qrels.setdefault(query_id, {})[doc_id] = int(row["score"])

    judged_queries = [
        Query(query_id=str(row["_id"]), text=row["text"])
        for row in query_rows
        if str(row["_id"]) in qrels
    ]

    sampled = max_queries is not None and len(judged_queries) > max_queries
    if sampled:
        judged_queries = judged_queries[:max_queries]
        kept_ids = {q.query_id for q in judged_queries}
        qrels = {qid: judged for qid, judged in qrels.items() if qid in kept_ids}
        judged_doc_ids = {doc_id for judged in qrels.values() for doc_id in judged}
        corpus_rows = [row for row in corpus_rows if str(row["_id"]) in judged_doc_ids]

    documents = tuple(
        Document(doc_id=str(row["_id"]), title=row.get("title", ""), text=row["text"])
        for row in corpus_rows
    )
    return RetrievalDataset(
        info=info,
        split=split,
        documents=documents,
        queries=tuple(judged_queries),
        qrels=qrels,
        sampled=sampled,
    )
```

Block by block:

1. **Qrels first**: we build the dictionary `query_id → {doc_id: score}`,
   with `str()` on each id — the remedy for the type pitfall seen above, applied at
   the system boundary (and if it leaks, the validator catches it).
2. **Query filtering**: only those present in the split's qrels survive
   (1,109 → 300). The guarantee required by check no. 3 of the validator.
3. **Sampling** (sample mode only): we keep the first `max_queries`
   queries — *first*, not drawn at random: the file order is
   stable, so two CI runs load exactly the same dataset — and only
   the documents they judge. Accepted consequence: the corpus no longer has
   "distractors" (the non-relevant documents among which to search), so a
   retrieval score on it would mean nothing. That is exactly what `sampled=True`
   declares.

## Part 5 — The rest of the testbench: fifteen datasets, each with its blind spot

SciFact only tests "retrieve a document". Each capability of the final system has its
dedicated dataset, chosen and **verified** (availability, size, license, pitfalls —
details in `docs/datasets-verification-2026-08-26.md` and ADR 001). Each family arrives
in the phase that needs it; nothing else is loaded today.

### Retrieval and embeddings (phases 2-3)

*Embeddings*: representation of a text as a vector of numbers, such that two texts
close in meaning give close vectors — the building block of "semantic" search,
which we will compare to keyword search (BM25).

| Dataset | What it provides | What it will measure | Verified pitfall |
|---|---|---|---|
| **SciFact** | 5.2k docs, 300 queries | validating the retrieval mechanics | qrels in a separate repository; unjudged queries to filter out |
| **NFCorpus** | 3.6k medical docs, **graded** qrels 0-2 | graded-relevance metrics | ~134k judgments: "relevant" is no longer binary there |
| **FiQA** | 57.6k finance docs | scaling ×10, jargon | 6× larger, slower iterations |
| **MTEB** | standardized protocol, 22 tasks in French | comparing embedding models (phase 1) | v2 API incompatible with all earlier tutorials |

### Question answering: multi-hop, refusal, long documents (phases 2-3)

| Dataset | What it provides | What it will measure | Verified pitfall |
|---|---|---|---|
| **HotpotQA** | 90k 2-hop questions + evidence passages | **multi-hop**: combining several documents | test answers hidden → evaluate on validation |
| **SQuAD 2.0** | 142k questions including ~50k **unanswerable** | knowing how to say "I don't know" (correct vs excessive refusal) | "unanswerable" is encoded by an empty list, not a boolean |
| **QASPER** | 1,585 **full** scientific papers | chunking long documents | loads only through a special branch of the HF repository |
| MuSiQue, 2WikiMultihopQA | harder multi-hop | in reserve | Google Drive/Dropbox distribution, unofficial mirrors |

**Natural Questions**, initially in the table, was **removed**: ~145 GB for an
"unanswerable" signal that SQuAD 2.0 provides in 46 MB (ADR 001).

### SQL, agents, security (phase 4)

*Agent*: an LLM that does not just answer, but chooses and chains
**tools** (SQL query, API call…) to accomplish a task. *Text-to-SQL*:
translating a question into a SQL query executed on a real database — the ground
truth is no longer a passage to retrieve but **the execution result**.

| Dataset | What it provides | What it will measure | Verified pitfall |
|---|---|---|---|
| **Spider** | 10k questions, 200 **SQLite databases included** | SQL execution accuracy | databases on Google Drive only; massive documented contamination |
| **BIRD Mini-Dev** | 500 questions, 11 databases (**PostgreSQL provided**) | SQL on large, dirty databases | full version: 33 GB — excluded |
| **tau2-bench** | complete simulated environments (tools + rules + simulated customer) | multi-turn agents | replaces τ-bench, **officially deprecated**; double cost (2 LLMs per exchange) |
| **BFCL v4** | finely annotated tool calls | exact arguments, parallel calls, "call nothing" | harness pinned to Python 3.10 → separate environment |
| **AgentDojo** | 629 **prompt injection** attacks via the data | agent security | unstable API (pre-1.0) |

### Tables, images, time series (phase 4)

| Dataset | What it provides | What it will measure | Verified pitfall |
|---|---|---|---|
| **TAT-QA** | 16.5k questions on financial reports, text+table | hybrid numerical reasoning | the cleanest of its family (17 MB) |
| WikiTableQuestions, FinQA | questions on tables, calculations | arithmetic by code, never by the LLM | FinQA: historical bug that inflated published scores |
| **DocVQA** | 50k questions on scanned documents | document reading (multimodal) | official site: registration + broken TLS → HF mirrors; unclear license |
| **ChartQA** | 32.7k questions on charts | chart reading | GPL-3.0 license, to be flagged |
| **CORD v2**, FUNSD | receipts and forms, labeled fields | field-by-field structured extraction | CORD: 1k published out of the paper's 11k; FUNSD: non-commercial license |
| Time series | **generated by us**, truth computed by pandas | aggregations, time zones, gaps | zero possible contamination — by construction |

## Part 6 — Your turn

Change the configuration at the very top, then "Run All":

1. **`SPLIT = "train"`** — the other SciFact split: other queries, other qrels,
   and (the next cell demonstrates it) **no overlap** with test.
2. **`SHOW_QUERY_INDEX = 1`, `2`, …** — other claims and their documents.
3. In sample mode: **`MAX_QUERIES = 2`** — observe `sampled: True` and the corpus
   reduced to the judged documents only.

In [12]:
# Test vs train: two disjoint sets of queries.
other_split = "train" if SPLIT == "test" else "test"
other = loader.load(other_split, max_queries=MAX_QUERIES if SAMPLE else None)

current_ids = {q.query_id for q in dataset.queries}
other_ids = {q.query_id for q in other.queries}
print(f"{SPLIT}: {dataset.describe()}")
print(f"{other_split}: {other.describe()}")
print(f"queries shared by both splits: {len(current_ids & other_ids)}")

test: {'dataset': 'beir/scifact', 'split': 'test', 'documents': 6, 'queries': 3, 'judgments': 4, 'sampled': False}
train: {'dataset': 'beir/scifact', 'split': 'train', 'documents': 6, 'queries': 1, 'judgments': 1, 'sampled': False}
queries shared by both splits: 0


In [13]:
# A loader rejects what it does not know — loudly, with the list of what it knows.
try:
    BeirLoader("nfcorpus")
except ValueError as error:
    print(f"ValueError: {error}")

ValueError: BEIR dataset not wired: 'nfcorpus' (known: ['scifact'])


## Interview questions

<details><summary><b>1. Why evaluate on public datasets rather than on your own corpus?</b></summary>

Published scores provide an objective reference point: an implementation far below
the literature is wrong, and you know it immediately. On an in-house corpus, no
reference point — and the annotation cost is huge. A trade-off to always mention:
contamination (these datasets are in the models' training data), hence inflated
scores; measured with a "fresh" comparison dataset. An in-house corpus remains useful
to demonstrate the *method* of building an evaluation (our golden set, phase 2).
</details>

<details><summary><b>2. What is a qrel? Binary vs graded — what does it change?</b></summary>

A relevance judgment: for a query, which documents are relevant, to what
degree. Binary (SciFact): counting the relevant documents retrieved is enough (recall,
MRR). Graded (NFCorpus, 0-2): you need a metric sensitive to the degree — typically
nDCG — and the threshold "relevant if ≥ 1" becomes a choice to document.
</details>

<details><summary><b>3. The queries file contains queries that were never judged. What happens if you do not filter them out?</b></summary>

Any metric averaged over the queries is diluted: a query without qrels scores 0 by
construction (nothing to retrieve), the average drops silently, without an error.
SciFact: 1,109 queries in the file, 300 judged in test — without filtering, every score
would be divided by ~3.7. Hence validation at construction: query without a judgment =
immediate exception.
</details>

<details><summary><b>4. How do you guarantee that a CI (sample) number will never be presented as a result?</b></summary>

The flag is carried by the data itself (`sampled=True`), set by the loader
at truncation time — not by a team convention or an environment
variable. The harness (phase 2) will refuse to archive a run on a sampled dataset.
Moreover, the sampled corpus no longer has distractors: any score on it would be
absurd, so it may as well be impossible to publish.
</details>

<details><summary><b>5. Why cast all ids to strings?</b></summary>

The sources are inconsistent: SciFact delivers the corpus ids as `str` and those of
the qrels as `int`. In Python, `4983 == "4983"` is false **silently**: no error,
just orphan qrels and a wrong recall. We normalize at the boundary (in the
join) and the integrity validation catches any residual leak.
</details>

<details><summary><b>6. Your SciFact scores beat the BEIR paper. Your reaction?</b></summary>

Suspicion first, celebration never: a large gap — in either direction — is
a bug until proven otherwise. Leads: split leakage (evaluation on train
queries), incorrectly implemented metric, contamination of the embedding model,
truncated corpus. Phase 2 adds bootstrap confidence intervals: without them, a
"+2%" has a good chance of not existing.
</details>

## What we did not do, and why

- **No metrics** (recall, nDCG…): that is the core of phase 2, the central phase
  of the project. Here we only deliver clean data to measure — and the vocabulary to
  talk about it.
- **The other loaders** (NFCorpus, HotpotQA, QASPER, Spider…): each phase plugs in
  the ones it needs. The contract (types + validation + fixtures + tests) is in place;
  the additions are mechanical.
- **No version pinning of the HF repositories**: the BeIR repositories have been stable
  for years, and the literature check would cry out at the first volume drift. We
  will pin if that happens.
- **Langfuse and the AWS cloud**: deferred to phase 1 (ADR 002) — nothing to trace and
  nothing to deploy as long as there is neither an LLM call nor an API.

**Next (phase 1)**: the production skeleton — robust LLM layer (retries,
timeouts, cache, streaming), sources interface, ingestion and first pgvector index,
`/chat` API deployed on the cluster. Then phase 2: measure everything.

## Summary glossary

| Term | Definition |
|---|---|
| **LLM** | Large Language Model — generative language model (GPT, Claude, Mistral…) |
| **RAG** | Retrieval-Augmented Generation: retrieve relevant passages, then generate an answer grounded in them |
| **Retrieval** | The "search engine" step: finding the right documents in a corpus |
| **Corpus** | The set of documents we search in |
| **Query** | A test request (question, keywords or claim depending on the dataset) |
| **Qrels** | *Query relevance judgments*: for each query, the documents judged relevant and their degree |
| **Ground truth** | The right answers, established in advance, against which we measure |
| **Benchmark** | Shared dataset + measurement protocol, making it possible to compare systems with one another |
| **The literature** | The scores published in research papers on these benchmarks — our yardstick |
| **Split** | Subset of a dataset (train to tune, test to measure — never the reverse) |
| **BEIR** | Standard collection of some fifteen retrieval datasets in a uniform format |
| **SciFact** | BEIR dataset: find the abstract that supports/refutes a scientific claim |
| **BM25** | Classic keyword search algorithm — the baseline to beat |
| **nDCG@10** | Retrieval metric (0-1) rewarding relevant documents placed at the top of the first 10 |
| **Embedding** | Vector representation of a text, the basis of semantic search |
| **Multi-hop** | Question that requires combining several documents to answer |
| **Contamination** | Presence of public datasets in the models' training data → inflated scores |
| **Fixture** | Tiny hand-made data file for testing offline |
| **Loader** | Code that downloads a dataset and turns it into validated project objects |
| **Pydantic** | Python validation library: an invalid object cannot be constructed |
| **Agent** | LLM that chooses and chains tools (SQL, API…) to accomplish a task |
| **Text-to-SQL** | Translating a question into a SQL query; truth = execution result |